# SciBERT Evaluation

Loads the fine-tuned checkpoint, evaluates on a held-out test set, and exports per-field metrics and a confusion matrix.

## Train / val / test split (80 / 10 / 10)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_idx']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label_idx']
)

print(f"train: {len(train_df):,} | val: {len(val_df):,} | test: {len(test_df):,}")

In [ ]:
test_dataset = PaperDataset(test_df['text'], test_df['label_idx'], tokenizer)

## Test set evaluation

In [ ]:
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader

MODEL_PATH = "scibert_classifier"
best_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE)

test_loader = DataLoader(test_dataset, batch_size=32)
_, preds, trues = evaluate(best_model, test_loader)
target_names = [id2label[i] for i in range(NUM_LABELS)]

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

acc = accuracy_score(trues, preds)
macro_f1 = f1_score(trues, preds, average='macro')
weighted_f1 = f1_score(trues, preds, average='weighted')

print(f"accuracy: {acc:.4f}")
print(f"macro F1: {macro_f1:.4f}")
print(f"weighted F1: {weighted_f1:.4f}\n")
print(classification_report(trues, preds, target_names=target_names, digits=3))

## Export per-field metrics

In [ ]:
import pandas as pd

report = classification_report(trues, preds, target_names=target_names, digits=3, output_dict=True)

df_report = pd.DataFrame([
    {
        "field": name,
        "precision": round(report[name]['precision'], 3),
        "recall": round(report[name]['recall'], 3),
        "f1": round(report[name]['f1-score'], 3),
        "support": int(report[name]['support']),
    }
    for name in target_names
]).sort_values('f1', ascending=False)

df_report.to_csv("scibert_per_class_f1.csv", index=False)
df_report

In [ ]:
import json

summary = {
    "test_set_size": len(trues),
    "accuracy": round(acc, 4),
    "macro_f1": round(macro_f1, 4),
    "weighted_f1": round(weighted_f1, 4),
    "num_classes": NUM_LABELS,
    "highest_f1_class": df_report.iloc[0]['field'],
    "highest_f1_score": df_report.iloc[0]['f1'],
    "lowest_f1_class": df_report.iloc[-1]['field'],
    "lowest_f1_score": df_report.iloc[-1]['f1'],
}

with open("scibert_eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

summary

## Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

SHORT_NAMES = {
    "Agricultural and Biological Sciences": "Agri & Bio",
    "Biochemistry, Genetics and Molecular Biology": "Biochem & Gen",
    "Chemical Engineering": "Chem Eng",
    "Chemistry": "Chemistry",
    "Computer Science": "Comp Sci",
    "Decision Sciences": "Decision Sci",
    "Dentistry": "Dentistry",
    "Earth and Planetary Sciences": "Earth Sci",
    "Energy": "Energy",
    "Engineering": "Engineering",
    "Environmental Science": "Env Sci",
    "Health Professions": "Health Prof",
    "Immunology and Microbiology": "Immuno/Micro",
    "Materials Science": "Materials",
    "Mathematics": "Mathematics",
    "Medicine": "Medicine",
    "Neuroscience": "Neuroscience",
    "Nursing": "Nursing",
    "Pharmacology, Toxicology and Pharmaceutics": "Pharma & Tox",
    "Physics and Astronomy": "Physics",
    "Veterinary": "Veterinary",
}
short_labels = [SHORT_NAMES.get(n, n) for n in target_names]

cm = confusion_matrix(trues, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f',
    xticklabels=short_labels, yticklabels=short_labels,
    cmap='Blues', linewidths=0.3, ax=ax,
)
ax.set_title('SciBERT confusion matrix (row-normalized, test set)', fontsize=13, pad=12)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('scibert_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Save results to Drive

In [ ]:
import os
import shutil

SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/scibert_eval_results"
os.makedirs(SAVE_DIR, exist_ok=True)

for fname in ["scibert_per_class_f1.csv", "scibert_eval_summary.json", "scibert_confusion_matrix.png"]:
    shutil.copy(fname, os.path.join(SAVE_DIR, fname))